# Simple Mammography Training with ResNet-50

**Train a breast cancer classification model using ResNet-50 backbone**

## Architecture:
- **Backbone**: ResNet-50 pretrained on ImageNet
- **Fine-tuning**: Partial (configurable fraction of layers)
- **Dropout**: Configurable dropout before classifier
- **Output**: Binary classification (sigmoid)

## Features:
- Patient-level train/val split (prevents data leakage)
- Early stopping based on validation PR-AUC
- Breast-level evaluation using Noisy-OR aggregation
- Simple, clean training loop

---

## Configuration

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION - Modify these values
# ============================================================================

# Paths
BASE_DIR = '/content/drive/MyDrive/vindr-mammo'
PREPROCESSED_DIR = f'{BASE_DIR}/preprocessed_png_512'  # From preprocessing pipeline
METADATA_CSV = f'{PREPROCESSED_DIR}/train.csv'  # Training metadata
VAL_METADATA_CSV = f'{PREPROCESSED_DIR}/val.csv'  # Validation metadata
OUTPUT_DIR = '/content/drive/MyDrive/training_output'

# Model hyperparameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DROPOUT_RATE = 0.3
UNFREEZE_FRACTION = 0.5  # Unfreeze last 50% of ResNet layers

# Data augmentation
AUGMENTATION_STRENGTH = 0.5  # 0.0 = no augmentation, 1.0 = maximum

# Training settings
BATCH_SIZE = 32
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
IMAGE_SIZE = (224, 224)  # ResNet-50 standard input

# Device
DEVICE = 'cuda'  # 'cuda' or 'cpu'

print("✅ Configuration loaded")

## Step 1: Setup Environment

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## Step 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")

## Step 3: Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib pillow tqdm
print("✅ Dependencies installed")

## Step 4: Load Dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

# Load metadata
print("📊 Loading dataset metadata...")
train_df = pd.read_csv(METADATA_CSV)
val_df = pd.read_csv(VAL_METADATA_CSV)

print(f"\n✅ Dataset loaded:")
print(f"  Train: {len(train_df)} images")
print(f"    Malignant: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).sum()/len(train_df)*100:.1f}%)")
print(f"    Benign:    {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).sum()/len(train_df)*100:.1f}%)")

print(f"\n  Val: {len(val_df)} images")
print(f"    Malignant: {(val_df['label'] == 1).sum()} ({(val_df['label'] == 1).sum()/len(val_df)*100:.1f}%)")
print(f"    Benign:    {(val_df['label'] == 0).sum()} ({(val_df['label'] == 0).sum()/len(val_df)*100:.1f}%)")

# Prepare metadata for model (add required columns if missing)
def prepare_metadata(df):
    """Ensure metadata has all required columns."""
    df = df.copy()
    
    # Create image_id if missing
    if 'image_id' not in df.columns:
        df['image_id'] = df['png_path'].apply(lambda x: Path(x).stem)
    
    # Create patient_id if missing (use study_id)
    if 'patient_id' not in df.columns and 'study_id' in df.columns:
        df['patient_id'] = df['study_id']
    
    # Create breast_id if missing
    if 'breast_id' not in df.columns:
        if 'laterality' in df.columns:
            df['breast_id'] = df['patient_id'] + '_' + df['laterality']
        else:
            df['breast_id'] = df['patient_id'] + '_Unknown'
    
    # Rename png_path to image_path if needed
    if 'png_path' in df.columns and 'image_path' not in df.columns:
        df['image_path'] = df['png_path']
    
    return df

train_df = prepare_metadata(train_df)
val_df = prepare_metadata(val_df)

print("\n✅ Metadata prepared")
print(f"   Columns: {list(train_df.columns)}")

## Step 5: Define Model Architecture

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

class ResNet50Classifier(nn.Module):
    """
    ResNet-50 with ImageNet pretrained weights and partial fine-tuning.
    
    Binary classification with dropout and sigmoid output.
    """
    
    def __init__(self, unfreeze_fraction=1.0, dropout_rate=0.0):
        """
        Args:
            unfreeze_fraction: Fraction of backbone layers to unfreeze (0.0 to 1.0)
            dropout_rate: Dropout rate before classification head
        """
        super().__init__()
        
        self.unfreeze_fraction = max(0.0, min(1.0, unfreeze_fraction))
        self.dropout_rate = dropout_rate
        
        # Load pretrained ResNet-50
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        
        # Remove original classification head
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        
        # Binary classification head with dropout
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(2048, 1)  # ResNet-50 has 2048 features
        
        # Apply partial fine-tuning
        self._setup_partial_finetuning()
    
    def _setup_partial_finetuning(self):
        """Freeze/unfreeze backbone layers based on unfreeze_fraction."""
        all_layers = list(self.features.children())
        n_layers = len(all_layers)
        n_unfreeze = int(n_layers * self.unfreeze_fraction)
        
        # Freeze all first
        for param in self.features.parameters():
            param.requires_grad = False
        
        # Unfreeze last n_unfreeze layers
        if n_unfreeze > 0:
            layers_to_unfreeze = all_layers[-n_unfreeze:]
            for layer in layers_to_unfreeze:
                for param in layer.parameters():
                    param.requires_grad = True
        
        # Classifier always trainable
        for param in self.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        """Forward pass."""
        features = self.features(x)
        features = self.avgpool(features)
        features = torch.flatten(features, 1)
        features = self.dropout(features)
        logits = self.classifier(features)
        probs = torch.sigmoid(logits).squeeze(1)
        return probs
    
    def get_trainable_params(self):
        """Get number of trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Create model
model = ResNet50Classifier(
    unfreeze_fraction=UNFREEZE_FRACTION,
    dropout_rate=DROPOUT_RATE
).to(DEVICE)

print("✅ Model created:")
print(f"   Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"   Trainable parameters: {model.get_trainable_params():,}")
print(f"   Unfreeze fraction: {UNFREEZE_FRACTION:.1%}")
print(f"   Dropout rate: {DROPOUT_RATE:.1%}")

## Step 6: Create Data Loaders

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

class MammogramDataset(Dataset):
    """Dataset for mammogram images."""
    
    def __init__(self, metadata, image_dir, transform=None, augmentation=None):
        self.metadata = metadata.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.augmentation = augmentation
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        
        # Load image
        image_path = os.path.join(self.image_dir, row['image_path'])
        image = Image.open(image_path).convert('L')  # Grayscale
        
        # Convert to tensor
        image = torch.from_numpy(np.array(image)).float() / 255.0
        image = image.unsqueeze(0)  # Add channel dim
        
        # Apply transforms
        if self.transform is not None:
            image = self.transform(image)
        
        # Convert to 3-channel for ResNet
        image = image.repeat(3, 1, 1)
        
        # Apply augmentation BEFORE normalization
        if self.augmentation is not None:
            image = self.augmentation(image)
        
        # ✅ Apply ImageNet normalization (CRITICAL for pretrained ResNet!)
        # ResNet-50 was pretrained with these specific values
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = (image - mean) / std
        
        label = int(row['label'])
        image_id = row['image_id']
        
        return image, label, image_id

# Define transforms
base_transform = transforms.Resize(IMAGE_SIZE)

# Simple intensity augmentation
class IntensityAugmentation:
    """Intensity-based augmentation for mammograms."""
    
    def __init__(self, strength=0.5):
        self.strength = strength
        self.brightness_factor = 0.2 * strength
        self.contrast_factor = 0.2 * strength
        self.noise_std = 0.05 * strength
    
    def __call__(self, x):
        if self.strength == 0.0:
            return x
        
        # Brightness
        brightness_delta = torch.rand(1).item() * self.brightness_factor * 2 - self.brightness_factor
        x = x + brightness_delta
        
        # Contrast
        contrast_delta = torch.rand(1).item() * self.contrast_factor * 2 - self.contrast_factor
        x = x * (1 + contrast_delta)
        
        # Noise
        noise = torch.randn_like(x) * self.noise_std
        x = x + noise
        
        # Clip to valid range
        x = torch.clamp(x, 0, 1)
        
        return x

# Create datasets
train_augmentation = IntensityAugmentation(strength=AUGMENTATION_STRENGTH)

train_dataset = MammogramDataset(
    metadata=train_df,
    image_dir=PREPROCESSED_DIR,
    transform=base_transform,
    augmentation=train_augmentation
)

val_dataset = MammogramDataset(
    metadata=val_df,
    image_dir=PREPROCESSED_DIR,
    transform=base_transform,
    augmentation=None  # No augmentation for validation
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("✅ Data loaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Augmentation strength: {AUGMENTATION_STRENGTH:.1%}")
print(f"\n✅ ImageNet normalization applied:")
print(f"   Mean: [0.485, 0.456, 0.406]")
print(f"   Std:  [0.229, 0.224, 0.225]")
print(f"   This matches ResNet-50 pretraining!")

## Step 7: Define Training Components

In [ ]:
import torch.optim as optim
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

# Loss function
criterion = nn.BCELoss()

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Early stopping
class EarlyStopping:
    """Early stopping based on validation metric."""
    
    def __init__(self, patience=15, mode='max'):
        self.patience = patience
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0
    
    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False
        
        improved = (score > self.best_score) if self.mode == 'max' else (score < self.best_score)
        
        if improved:
            self.best_score = score
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        
        return self.early_stop

early_stopping = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, mode='max')

# Metrics computation
def compute_metrics(predictions, labels):
    """Compute classification metrics."""
    predictions = np.array(predictions)
    labels = np.array(labels)
    
    metrics = {
        'auroc': roc_auc_score(labels, predictions),
        'pr_auc': average_precision_score(labels, predictions),
        'brier': brier_score_loss(labels, predictions)
    }
    
    return metrics

# Breast-level aggregation using Noisy-OR
def aggregate_to_breast_level(image_predictions, metadata):
    """Aggregate image-level predictions to breast-level using Noisy-OR."""
    breast_preds = {}
    breast_labels = {}
    
    for idx, row in metadata.iterrows():
        breast_id = row['breast_id']
        image_id = row['image_id']
        
        if image_id in image_predictions:
            pred = image_predictions[image_id]
            
            if breast_id not in breast_preds:
                breast_preds[breast_id] = []
                breast_labels[breast_id] = int(row['label'])
            
            breast_preds[breast_id].append(pred)
    
    # Apply Noisy-OR: 1 - prod(1 - p_i)
    final_preds = []
    final_labels = []
    
    for breast_id in breast_preds:
        preds = breast_preds[breast_id]
        noisy_or = 1.0 - np.prod([1.0 - p for p in preds])
        final_preds.append(noisy_or)
        final_labels.append(breast_labels[breast_id])
    
    return final_preds, final_labels

print("✅ Training components ready:")
print(f"   Optimizer: AdamW")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Weight decay: {WEIGHT_DECAY}")
print(f"   Early stopping patience: {EARLY_STOPPING_PATIENCE}")

## Step 8: Training Loop

In [ ]:
from tqdm.notebook import tqdm

# Training history
history = {
    'train_loss': [],
    'val_pr_auc': [],
    'val_auroc': [],
    'val_brier': []
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
best_checkpoint_path = os.path.join(OUTPUT_DIR, 'best_model.pt')

print("🚀 Starting training...\n")
print("="*70)

for epoch in range(MAX_EPOCHS):
    # ========== TRAINING ==========
    model.train()
    train_loss = 0.0
    n_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]")
    for images, labels, _ in pbar:
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE)
        
        # Forward
        predictions = model(images)
        loss = criterion(predictions, labels)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        n_batches += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = train_loss / n_batches
    history['train_loss'].append(avg_train_loss)
    
    # ========== VALIDATION ==========
    model.eval()
    image_predictions = {}
    image_labels = {}
    
    with torch.no_grad():
        for images, labels, image_ids in tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]  "):
            images = images.to(DEVICE)
            predictions = model(images).cpu().numpy()
            
            for img_id, pred, label in zip(image_ids, predictions, labels.numpy()):
                image_predictions[img_id] = float(pred)
                image_labels[img_id] = int(label)
    
    # Aggregate to breast-level
    breast_predictions, breast_labels = aggregate_to_breast_level(
        image_predictions, val_df
    )
    
    # Compute metrics
    val_metrics = compute_metrics(breast_predictions, breast_labels)
    
    history['val_pr_auc'].append(val_metrics['pr_auc'])
    history['val_auroc'].append(val_metrics['auroc'])
    history['val_brier'].append(val_metrics['brier'])
    
    # Print progress
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val PR-AUC: {val_metrics['pr_auc']:.4f}")
    print(f"  Val AUROC:  {val_metrics['auroc']:.4f}")
    print(f"  Val Brier:  {val_metrics['brier']:.4f}")
    print(f"  Breasts evaluated: {len(breast_labels)}")
    
    # Save best checkpoint
    if early_stopping.best_score is None or val_metrics['pr_auc'] > early_stopping.best_score:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
            'hyperparameters': {
                'learning_rate': LEARNING_RATE,
                'weight_decay': WEIGHT_DECAY,
                'dropout_rate': DROPOUT_RATE,
                'unfreeze_fraction': UNFREEZE_FRACTION,
                'augmentation_strength': AUGMENTATION_STRENGTH
            }
        }, best_checkpoint_path)
        print(f"  💾 Saved best checkpoint (PR-AUC: {val_metrics['pr_auc']:.4f})")
    
    # Check early stopping
    if early_stopping(val_metrics['pr_auc'], epoch):
        print(f"\n⚠️  Early stopping triggered at epoch {epoch+1}")
        print(f"   Best epoch: {early_stopping.best_epoch + 1}")
        print(f"   Best PR-AUC: {early_stopping.best_score:.4f}")
        break
    
    print("="*70)

print("\n✅ Training complete!")
print(f"   Best checkpoint saved to: {best_checkpoint_path}")
print(f"   Best epoch: {early_stopping.best_epoch + 1}")
print(f"   Best PR-AUC: {early_stopping.best_score:.4f}")

## Step 9: Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Training History', fontsize=16, fontweight='bold')

# Training loss
axes[0, 0].plot(history['train_loss'], color='blue', linewidth=2)
axes[0, 0].set_title('Training Loss', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('BCE Loss')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best Epoch')
axes[0, 0].legend()

# Validation PR-AUC
axes[0, 1].plot(history['val_pr_auc'], color='green', linewidth=2)
axes[0, 1].set_title('Validation PR-AUC', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('PR-AUC')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best Epoch')
axes[0, 1].legend()

# Validation AUROC
axes[1, 0].plot(history['val_auroc'], color='orange', linewidth=2)
axes[1, 0].set_title('Validation AUROC', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('AUROC')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best Epoch')
axes[1, 0].legend()

# Validation Brier Score
axes[1, 1].plot(history['val_brier'], color='purple', linewidth=2)
axes[1, 1].set_title('Validation Brier Score', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Brier Score (lower is better)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axvline(early_stopping.best_epoch, color='red', linestyle='--', label='Best Epoch')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training history saved to: {OUTPUT_DIR}/training_history.png")

## Step 10: Load Best Model and Final Evaluation

In [ ]:
# Load best checkpoint
checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print("✅ Best model loaded")
print(f"   Epoch: {checkpoint['epoch'] + 1}")
print(f"\n📊 Best Validation Metrics:")
print(f"   PR-AUC: {checkpoint['metrics']['pr_auc']:.4f}")
print(f"   AUROC:  {checkpoint['metrics']['auroc']:.4f}")
print(f"   Brier:  {checkpoint['metrics']['brier']:.4f}")

print(f"\n⚙️  Hyperparameters:")
for key, value in checkpoint['hyperparameters'].items():
    print(f"   {key}: {value}")

## Summary

✅ **Training complete!**

**Outputs:**
- Best model checkpoint: `{OUTPUT_DIR}/best_model.pt`
- Training history plot: `{OUTPUT_DIR}/training_history.png`

**Architecture:**
- ResNet-50 pretrained on ImageNet
- Partial fine-tuning with configurable fraction
- Dropout regularization
- Binary classification with sigmoid

**Next steps:**
- Evaluate on test set
- Test on INbreast dataset (transfer learning)
- Adjust hyperparameters and retrain if needed